# Lab 20: Large Language Models (LLM) on Jetson

**Duration:** 2 hours 30 minutes. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
ChatGPT runs in a big data centre. In this lab a **large language model (LLM)**, small enough for the Jetson, runs on **your Jetson**, with no internet and no account. It is much smaller than ChatGPT, so it is less clever, but it is private, free to use and always available. You start the engine yourself (**llama.cpp**, the program `llama-server`), load a **model file**, chat with it in the browser, measure how fast it writes (**tokens per second**) and shape its character with a **system prompt**. This model reads and writes **text only**. In Lab 21 it gets eyes.

**By the end of this lab you can:**
1. Explain what a language model, a token, the context window and a GGUF model file are.
2. Start `llama-server` yourself with a model file, and explain its main settings.
3. Chat with the model in the browser and in Python, and measure the time to the first word and the tokens per second.
4. Keep a conversation going by sending the history, and keep it inside the context window.
5. Change the answers with the system prompt and the temperature.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Press **Ctrl + S** often. The chat cells print the answer word by word. The next lab (Lab 23) makes the model answer from your own notes (RAG).

> **Memory.** The model uses about 3.2 GB of the 8 GB memory and it is **shared** by all students of this Jetson: **one** server is started and everybody uses it. If everyone also runs a camera model at the same time, the memory runs out. Close the live views of other labs before you start.
> **Be critical.** A small model sounds sure even when it is wrong. Never trust it for facts, numbers or safety.

## 2. Background

**Language model.** A language model is a neural network that learned, from a huge amount of text, to guess **the next word**. It writes an answer one piece at a time: it guesses a piece, adds it to the text, and guesses the next one. There is no database of facts inside, only patterns of words. This is why it can write fluent text that is wrong.

**Tokens.** The model does not see letters or words but **tokens**: pieces of words. "Jetson" can be two tokens, a short common word is one token. In English one token is about 4 characters or three-quarters of a word.

**Parameters.** The size of a model is the number of numbers it learned, written in billions (B). The models of this lab are small enough for the 8 GB memory of the Jetson:

| Model | Parameters | Memory (4-bit) | Character |
|---|---|---|---|
| `Qwen3.5-2B` | 2 B | about 1.3 GB (+ 0.7 GB for the eyes) | Our model: fast, many languages, and it can see pictures (Lab 21) |
| `LFM2.5-VL-1.6B` | 1.6 B | about 0.7 GB (+ 0.6 GB for the eyes) | The smallest, very fast |

The models are **quantised** to 4 bits (like the FP16 of Lab 15, but smaller): each number is stored in 4 bits instead of 16, so the model is four times smaller and runs faster, with a small loss of quality.

**GGUF.** A model is a file. The format of llama.cpp is **GGUF**: one file with the numbers (the weights) and the tokenizer. The name tells you the content: `Qwen3.5-2B-Q4_K_M.gguf` is the family *Qwen3.5*, **2 B** parameters, quantised to 4 bits (**Q4_K_M**).

**llama.cpp.** `llama-server` (from the open-source project llama.cpp) loads the model file into the GPU of the Jetson and gives it to your programs through a small web service on `http://127.0.0.1:8080`. Your notebook sends a question (an HTTP request) and gets the answer back piece by piece. It also has a **chat page** that any laptop can open in a browser. **One** student per Jetson starts the server; the other students of that Jetson use the same one, from their own notebooks and from the chat page.

**Speed.** Two numbers matter:
- **Time to first token**: how long until the first word appears (the model reads your question first).
- **Tokens per second**: how fast the answer is written. People read about 5 tokens per second, so 20 or more feels fast.

**The context window.** The model can only look at a limited number of tokens at once (here 4096, about 3000 words). A chat is not remembered by the model: your program sends the **whole conversation** each time. When it gets too long, the oldest messages must be dropped.

**The system prompt and the temperature.** The **system prompt** is a first message that tells the model how to behave ("You are a short and friendly lab helper"). The **temperature** (0 to 2) controls how random the next-word choice is: 0 is steady (always the most likely word), 1 is varied, 1.5 or more is wild.

**Hallucination.** When the model does not know, it still writes something that sounds right: a **hallucination**. In Lab 23 you reduce it by giving the model notes to answer from (RAG).

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(20)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Run llama.cpp Yourself (50 min)

### Step 1: Who Starts the Server?
The whole team of one Jetson shares **one** language model server. Your teacher tells you **which student starts it** (the other students do not start it). Every student runs the cell below, to see if the server is already running.

In [ ]:
import csv
import glob
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime

import labview

print("This notebook is saved in:", os.getcwd())
print("Language model server running:", labview.llm_ready())

### Step 2: The Model File
The model is one file in the folder `~/models`. Run the cell to see the files on this Jetson. Look at the **name** and the **size** of the file (the name has the family, the parameters and the quantisation, as in the Background).

In [ ]:
!ls -lh ~/models

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The name of the model file, and what its parts mean (family, parameters, quantisation):
- The size of the file in GB. Why is it smaller than 2 B numbers of 16 bits (4 GB)?:



### Step 3: Download the Model (only the ONE student that the teacher chose)
If the model file is not on this Jetson yet, the chosen student downloads it (about 1.3 GB, a few minutes, only the first time). The cell skips a file that is already there. **The other students do not run this cell:** go to Step 4 and wait.

In [ ]:
labview.llm_download()

### Step 4: Start `llama-server` in a Terminal (only the ONE student that the teacher chose)
In JupyterLab choose **File, New, Terminal**. In the terminal paste this line and press **Enter**:

```
~/llama.cpp/build/bin/llama-server -m ~/models/Qwen3.5-2B-Q4_K_M.gguf -ngl 99 -c 4096 -fa on -lm none --jinja --host 0.0.0.0 --port 8080
```

Wait about 20 seconds until the last lines say that the server is listening. **Leave the terminal open**: closing it stops the server. The settings:

| Setting | Meaning |
|---|---|
| `-m ...gguf` | the model file to load |
| `-ngl 99` | put all (up to 99) layers of the model on the GPU |
| `-c 4096` | the context window: 4096 tokens |
| `-fa on` | flash attention: less memory for the same answer |
| `-lm none` | load the file straight into the GPU memory (the default can run out of memory on this board) |
| `--jinja` | use the chat template of the model (how system, user and assistant messages are written) |
| `--host 0.0.0.0` | any computer in the network may connect (`127.0.0.1` would mean only this Jetson) |
| `--port 8080` | the door number of the web service |

> **In the next labs** the helper `labview.llm_start()` runs this same command for you, so nobody needs a terminal.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which setting decides how much text the model can look at once?:
- What would happen if you wrote  --host 127.0.0.1  ? Who could not connect?:



### Step 5: Join the Server (all students)
When the chosen student has started the server, every student runs the cell. It shows that the server answers, the link of the chat page, and what the server tells about itself.

In [ ]:
print("Server ready:", labview.llm_ready())
print("Chat page (open it in the browser of your laptop):", labview.llm_link())
props = labview._props()
print("Model file:", props.get("model_path"))
print("Can it see pictures?", props.get("modalities", {}).get("vision"))

### Step 6: Chat in the Browser
Open the **chat page** on your laptop (the link above). Ask three different questions: a fact, a task ("write a poem about the Jetson") and a question about itself ("which model are you?"). Look at the speed that the page shows.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your three questions and a short note on each answer:
- The speed (tokens per second) shown by the page:
- Ask it to describe a picture or look at the camera. What does it say, and why (Hint: the line 'Can it see pictures?')?:



Look at the memory. The GPU and the CPU **share** the 8 GB, so the model counts against it.

In [ ]:
!free -m

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model you use:
- Memory used and available (the free -m numbers):
- Why does only one student per Jetson start the server, and how do the others use it?:



---
# Part B: Your First Chat (35 min)

### Step 1: Ask a Question
`labview.llm_chat` sends a list of messages to the model and returns the answer and the speed. `on_token=print_piece` prints every piece of the answer as it arrives. The numbers are the **time to the first token** and the **tokens per second**.

In [ ]:
def print_piece(piece):
    print(piece, end="", flush=True)


def ask_once(question, **options):
    """One question, no history. Prints the answer word by word and the speed."""
    result = labview.llm_chat([{"role": "user", "content": question}], options=options, on_token=print_piece)
    print(f"\n\n[first word after {result['first_token_s']:.1f} s | {result['tokens']} tokens | {result['tokens_per_s']:.1f} tokens per second | total {result['total_s']:.1f} s]")
    return result


result = ask_once("In two sentences, what is a Jetson Orin Nano?")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the answer make sense? Was anything wrong?:
- Time to the first token:
- Tokens per second:
- Is the speed fast enough to read comfortably (about 5 tokens per second)?:


### Step 2: Look at the Request
Your notebook sent a list of messages. Look at the code of the function (it is short) to see how it talks to llama-server.

In [ ]:
import inspect
print(inspect.getsource(labview.llm_chat))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which web address does the program send the request to?:
- What does `stream` do? What would you see without it?:


### Step 3: A Long Question Is Slower to Start
Before it writes the first word, the model must **read** your whole question. A long question takes longer to start. Then `llama-server` **remembers** what it has already read, so the same question asked again starts quickly. The cell sends a short question once, then a long one (about 600 tokens) three times. Compare the **time to the first word**.

In [ ]:
LONG = "Notes: " + "The Jetson shares its memory between the CPU and the GPU. " * 40 + "Summarise the notes in one sentence."

result = labview.llm_chat([{"role": "user", "content": "Say hello in five words."}])
print(f"Short question:  first word {result['first_token_s']:.2f} s   {result['tokens_per_s']:.1f} tokens per second")
for turn in range(3):
    result = labview.llm_chat([{"role": "user", "content": LONG}], options={"max_tokens": 40})
    print(f"Long question {turn + 1}: first word {result['first_token_s']:.2f} s   {result['tokens_per_s']:.1f} tokens per second")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time to the first word: short question, and the three long questions:
- Why is the long question slower the first time, and quicker the next times?:


---
# Part C: Conversation, Character and Temperature (45 min)

### Step 1: A Conversation with Memory
The model remembers nothing. To make a conversation, your program keeps a **list of all messages** and sends the whole list every time. The function `chat` adds your message and the answer to `history`. The function `trim` drops the oldest messages when the history is longer than `BUDGET` tokens (about 4 characters per token), so it stays inside the context window.

In [ ]:
BUDGET = 1200                   # tokens of history we allow (the window is 4096, the answer needs room too)
SYSTEM = "You are a friendly lab helper. Answer in one or two short sentences."
history = [{"role": "system", "content": SYSTEM}]


def tokens_of(messages):
    """A rough count of the tokens of some messages: about 4 characters per token."""
    return sum(len(m["content"]) for m in messages) // 4


def trim(messages, budget=BUDGET):
    """Drop the oldest messages (not the system prompt) until the history fits in the budget."""
    messages = list(messages)
    while tokens_of(messages) > budget and len(messages) > 2:
        del messages[1]
    return messages


def chat(question, show=True, **options):
    """Add the question to the history, ask the model with the trimmed history and keep the answer."""
    global history
    history.append({"role": "user", "content": question})
    history = trim(history)
    result = labview.llm_chat(history, options=options, on_token=print_piece if show else None)
    history.append({"role": "assistant", "content": result["text"]})
    if show:
        print(f"\n[{result['tokens_per_s']:.1f} tokens per second | history about {tokens_of(history)} tokens]")
    return result


chat("My name is Aminah. Please remember it.")
chat("What is my name?")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did it remember your name? Why can it, when the model has no memory?:
- What happens to the history size after each message?:


Now clear the history (a new chat) and ask the same question. The model has forgotten.

In [ ]:
history = [{"role": "system", "content": SYSTEM}]
chat("What is my name?")

### Step 2: Check `trim` without the Model
The cell makes a long fake history and shows that `trim` keeps the system prompt and drops the oldest messages.

In [ ]:
fake = [{"role": "system", "content": "system"}] + [{"role": "user", "content": f"message {i} " + "word " * 200} for i in range(20)]
print("Before:", len(fake), "messages, about", tokens_of(fake), "tokens")
short = trim(fake, budget=1200)
print("After: ", len(short), "messages, about", tokens_of(short), "tokens")
print("First message kept:", short[0]["content"], "| oldest user message kept:", short[1]["content"][:12])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Messages before and after `trim`:
- Which messages are dropped and which are kept? Is that a good rule?:


### Step 3: The System Prompt Changes the Character
The cell asks the **same** question with three different system prompts. Each time it starts with a fresh history.

In [ ]:
PERSONAS = {
    "short":   "Answer in one short sentence.",
    "teacher": "You are a patient teacher for 15-year-old students. Explain simply with one example.",
    "pirate":  "You are a pirate. Answer in pirate language, in two sentences.",
}
for name, system in PERSONAS.items():
    print(f"--- {name}: {system}")
    labview.llm_chat([{"role": "system", "content": system}, {"role": "user", "content": "What is a GPU?"}], on_token=print_piece)
    print()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How did the three answers differ?:
- Write a system prompt of your own (a character or a rule) and what it did.:


### Step 4: The Temperature
Ask the same question three times at temperature 0 and three times at temperature 1.5. At 0 the answers should be the same. At 1.5 they vary, and may be strange. `max_tokens` limits the length of the answer in tokens.

In [ ]:
for temperature in (0.0, 1.5):
    print(f"=== temperature {temperature}")
    for turn in range(3):
        result = labview.llm_chat([{"role": "user", "content": "Give me a name for a robot cat. Only the name."}], options={"temperature": temperature, "max_tokens": 12})
        print("  ", result["text"])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Answers at temperature 0:
- Answers at temperature 1.5:
- When do you want a low temperature and when a high one? Give one example of each.:


---
# Part D: Compare and Report (20 min)

### Step 1: Make It Your Own
Do **at least one** of these, or one of your own. Change the code in the cells above, run that cell again, then try again.

- Write a **system prompt of your own** (a character: a pirate, a strict teacher, a Malay translator) and ask it 3 questions.
- Ask the same question at temperature 0, 0.7 and 1.5 and compare the answers in a table.
- Switch the server to another model (`labview.llm_start("lfm2.5-vl-1.6b")`; tell your team first, it restarts for everybody), ask the same 3 questions and compare the speed, the memory and the quality in a table.
- Answer in **Malay** (system prompt: *Jawab dalam Bahasa Malaysia*) and test the quality.
- Ask a calculation (`What is 17 times 23?`, the correct answer is 391) and a question about this class. Is the answer right? What does this teach you about language models?

> **Tip.** Change one thing at a time.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw (speed, memory or quality):
- One problem you found and how you fixed it:



### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: show the chat page, the speed, a conversation, a persona and the temperature.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model used, time to first token and tokens per second:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- What the model does well and badly:
- One idea to make it better in future:



> **Where next.** In Lab 23 you give the model **notes** so that it answers from facts instead of making things up (RAG). In Lab 21 it gets **eyes**: you show it a picture from the camera. In Lab 22 you combine **ears, eyes, brain and mouth** into a multi-modal assistant.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | `llama-server` runs (started in the terminal by the chosen student) and you can explain `-m`, `-ngl`, `-c`, `-lm none` and `--host` | |
| 2 | The server answers in the notebook, and in the browser of your laptop, with the speed shown | |
| 3 | The short and the long question times are compared and explained | |
| 4 | The conversation remembers your name and the history is trimmed correctly | |
| 5 | The system prompt changes the character of the answers | |
| 6 | The effect of the temperature is shown with 3 answers each | |
| 7 | At least one improvement works | |
| 8 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What does a language model do when it writes an answer? Why can it write something that is wrong?:
- 2. What is a token? Roughly how many tokens are 100 words?:
- 3. What is in a GGUF file? What do `2B` and `Q4_K_M` in the name mean?:
- 4. What does `-ngl 99` do? What would happen with `-ngl 0`?:
- 5. Why must the program send the whole conversation each time?:
- 6. What is the context window, and what does `trim` do about it?:
- 7. What does the temperature change? Which value would you use for a bot that gives exact facts?:
- 8. Name one advantage and one disadvantage of a small model on the Jetson, compared with a big model in the cloud.:
- 9. Which task is a language model bad at? (a) Writing a polite email (b) Exact calculations with big numbers (c) Rewriting a text in simple words (d) Telling you what happened in the news this morning:



## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Server ready: False` | The chosen student has not started the server yet | Wait, or ask the teacher which student starts it (Step 4 of Part A) |
| `Cannot reach the language model ...` | The server is not running, or its terminal was closed | The chosen student starts it again (Step 4), or runs `labview.llm_start()` |
| The terminal says `out of memory` or `failed to allocate` | Other programs use the 8 GB, or the file cache is full | Close other notebooks (Kernel, Shut Down All Kernels), then start again with `-lm none` |
| `No such file` for the model | The model is not downloaded | The chosen student runs Step 3 |
| The download is very slow or stops | Slow internet, or another student downloads | Wait. Only one student needs to download. |
| The first answer takes a long time | A long question must be read first, or the server has just started | Normal. The next answers are quicker. |
| The server restarted while I worked | A team mate changed the model | Normal: run the cell again. Ask which model they chose (`labview.llm_models()`). |
| It is very slow (below 5 tokens per second) | The model runs on the CPU, or memory is nearly full | Close other notebooks. Check the power mode (Lab 15). |
| The answer is cut | `max_tokens` is too small | Raise it, or remove it from the options |
| The answer is wrong or made up | A small model hallucinates | Never trust it for facts. Lab 23 gives it notes. Lower the temperature. |
| `input` does not appear | The cell was already running | Press Stop in the toolbar and run the cell again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(20)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- llama.cpp: https://github.com/ggml-org/llama.cpp
- llama-server (API and chat page): https://github.com/ggml-org/llama.cpp/tree/master/tools/server
- The GGUF format: https://github.com/ggml-org/ggml/blob/master/docs/gguf.md
- Qwen3.5: https://huggingface.co/Qwen/Qwen3.5-2B
- Tokenizer visualiser (OpenAI): https://platform.openai.com/tokenizer